# Gemma title debug

Set `NOTE_PATH`, then run the cells. This loads the local Gemma 3 270M snapshot and uses the app's current production title prompt and generation settings.

In [25]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
REPO_ROOT = next(parent for parent in (cwd, *cwd.parents) if (parent / 'Nts' / 'FilenameModelConfiguration.swift').is_file())
EVALUATOR = REPO_ROOT / 'Developer' / 'TitleEvaluation'
sys.path.insert(0, str(EVALUATOR))

from title_eval.execution import messages_for
from title_eval.production import production_configuration
from mlx_lm import load, stream_generate
from mlx_lm.sample_utils import make_sampler
import mlx.core as mx

production = production_configuration(REPO_ROOT)
prompt = production['prompt']
generation = production['generation']
MODEL_DIR = REPO_ROOT / 'benchmarks' / 'title_generation' / 'models' / 'smollm2-1.7b-instruct-q8-mlx'
# MODEL_DIR = REPO_ROOT / 'benchmarks' / 'title_generation' / 'models' / 'qwen3-0.6b-4bit'
# MODEL_DIR = REPO_ROOT / 'benchmarks' / 'title_generation' / 'models' / 'gemma-3-270m-it-4bit'
assert MODEL_DIR.is_dir(), f'Gemma snapshot not found: {MODEL_DIR}'

In [26]:
model, tokenizer = load(str(MODEL_DIR), tokenizer_config={'trust_remote_code': False})
mx.eval(model.parameters())
print('Loaded Gemma 3 270M. Production prompt:', prompt['version'])

Loaded Gemma 3 270M. Production prompt: title-v2


In [32]:
# Example from the local Obsidian snapshot; change this path to try another file.
NOTE_PATH = REPO_ROOT / "benchmarks/title_generation/data/private/obsidian-2026-10-26/files/dp6/06 - Tutoria/Reuniões/Próxima reunião.md"
assert NOTE_PATH.is_file(), f"Example note not found: {NOTE_PATH}"
note_text = NOTE_PATH.read_text(encoding='utf-8')
messages = messages_for(note_text, prompt, generation)
rendered_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=generation.get('enable_thinking', False),
)
sampler = make_sampler(temp=generation['temperature'], top_p=generation['top_p'], top_k=0)
chunks = []
for chunk in stream_generate(
    model, tokenizer, prompt=rendered_prompt,
    max_tokens=generation['max_output_tokens'], sampler=sampler,
):
    chunks.append(chunk.text)
raw_title = ''.join(chunks)
print(raw_title)

"Dúvidas de Projeto"


In [24]:
# Free model memory when finished.
model = None
tokenizer = None
mx.clear_cache()